# Fitting a radial-velocity orbit, the open way

This tutorial fits a single-lined spectroscopic (SB1) radial-velocity
orbit to synthetic data in the spirit of the
[emcee line-fitting tutorial](https://emcee.readthedocs.io/en/stable/tutorials/line/):
everything is explicit, editable, and close to the data. We progress
from a quick periodogram, to a closed-form least-squares baseline, to a
quick maximum-likelihood point, to a hand-written probability and a bare
`emcee` run.

We sample **only the non-linear orbit shape** `theta = (P, e, tau)` and
recover the linear amplitudes `(K, omega, gamma)` per posterior draw --
the same scheme as `fit_astrometric_orbit.ipynb` and
`fit_joint_orbit.ipynb`. We never touch the production engine's internal
latent reparametrization. We only *call* audited, public functions from
`orblet` for the forward model and the linear solve, plus standard
`numpy`, `scipy`, `astropy`, `emcee`, and `corner`.

**Units and conventions** (reused from the audited model, not
re-derived):

- `P` orbital period in **days**; internally converted to Keplerian
  years (`P_yr = P / 365.25`) at the model interface.
- `e` eccentricity, dimensionless, `0 <= e < 1`.
- `tau` periastron phase fraction in `[0, 1)`; `tp = tau * P + t_ref`.
- `K_kms` primary radial-velocity semi-amplitude in **km/s** (recovered).
- `omega` argument of periastron in **radians**, in the **primary**
  frame (the binary-star / textbook convention the model encodes):
  `v = gamma + K [cos(nu + omega) + e cos(omega)]` (recovered).
- `gamma` systemic velocity offset in **km/s** (recovered).

Radial-velocity data alone do not constrain inclination; the audited
model fixes `sin(i) = 1`, so the fitted amplitude `K_kms` measures the SB1
mass function, not the true companion mass.

> **Which fit is this?** This is the simple **quick-look / linearized baseline**: a closed-form linear solve at a fixed orbital shape, a `scipy` maximum-likelihood refinement, and an optional bare `emcee` run, all written out in the open. The all-parameter version, every parameter sampled, is `fit_rv_orbit_all_parameters.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from astropy.timeseries import LombScargle
import emcee
import corner

# Audited, public building blocks. We use ONLY the forward model and
# the Gaussian likelihood atom -- NOT the production samplers -- so the
# probability we sample is written out explicitly below.
from orblet.simulate.bundles import load_simulated_inputs
from orblet.model import rv_model
from orblet import (
    semi_amplitude_kms,
    rv_design_matrix,
    linear_solve_rv,
    recover_K,
    recover_omega,
)

rng = np.random.default_rng(0)
DAYS_PER_YEAR = 365.25  # Keplerian year, matches the audited model

## 1. The data

We load the synthetic demo bundle (the toy orbit, strongly detected in both channels) and
pull out the radial-velocity epochs. The bundle is fully in-memory and
synthetic -- no files are read, no real targets are touched.

The observation times are in **days from J2010.0**; we adopt that same
scale as our reference epoch (`T_REF = 0.0`) so that `tp = tau * P`.

In [ ]:
bundle = load_simulated_inputs(seed=0)  # the toy orbit on the demo cadence
rv = bundle.rv_data

valid = np.asarray(rv['rv_validity_flag'])
t = np.asarray(rv['obs_time_rv'], dtype=float)[valid]        # days from J2010.0
v = np.asarray(rv['radial_velocity'], dtype=float)[valid]     # km/s
verr = np.asarray(rv['radial_velocity_err'], dtype=float)[valid]  # km/s

T_REF = 0.0  # reference epoch (days), same scale as the obs times
# Total system mass is an input to the audited K<->mass map; it is fixed
# here (RV alone does not separate the masses). Edit freely.
M_TOTAL_MSUN = 10.5

print(f'{t.size} RV epochs over {t.min():.0f}..{t.max():.0f} days')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(t, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2)
ax.set_xlabel('time (days from J2010.0)')
ax.set_ylabel('radial velocity (km/s)')
ax.set_title('Synthetic SB1 radial velocities')
plt.show()

## 2. Period search (Lomb-Scargle)

A first look at the dominant period. The Lomb-Scargle periodogram is a
good starting point even though the orbit is eccentric (the true RV
curve is not a pure sinusoid, so the peak is only an estimate).

In [ ]:
ls = LombScargle(t, v, verr)
freq, power = ls.autopower(
    minimum_frequency=1.0 / 500.0,   # search 10..500 day periods
    maximum_frequency=1.0 / 10.0,
    samples_per_peak=20,
)
periods = 1.0 / freq
pbest = float(periods[np.argmax(power)])
print(f'best Lomb-Scargle period: {pbest:.2f} days')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(periods, power, color='k', lw=1)
ax.axvline(pbest, color='C3', ls='--', label=f'pbest = {pbest:.1f} d')
ax.set_xlabel('period (days)')
ax.set_ylabel('Lomb-Scargle power')
ax.legend()
plt.show()

## 3. Closed-form least-squares baseline

At a **fixed** non-linear shape `(P, e, tau)`, the Keplerian RV model is
*linear* in the three amplitudes `(gamma, K cos(omega), K sin(omega))`:

$$ v = \gamma\cdot 1 + (K\cos\omega)(\cos\nu + e) + (K\sin\omega)(-\sin\nu). $$

So we get `K_kms`, `omega`, and `gamma` from a single 3x3 generalised
least-squares solve -- no sampling needed. We use the audited
`linear_solve_rv` core (with its `_rv_design_matrix`) at the
Lomb-Scargle period and a guessed eccentricity, scanning `tau`.

In [ ]:
def linear_amplitudes(P_days, e, tau):
    """Closed-form (K, omega, gamma) at fixed shape via the audited GLS core."""
    X = rv_design_matrix(t, period_yr=P_days / DAYS_PER_YEAR, ecc=e,
                          tau=tau, epoch_ref_mjd=T_REF)
    sol = linear_solve_rv(v, verr, X)
    K = recover_K(sol.beta)
    omega = recover_omega(sol.beta) % (2.0 * np.pi)
    gamma = float(sol.beta[0])
    return K, omega, gamma, sol

# Scan tau at (P=pbest, e=0.3) for the best-fitting linear amplitudes.
e_guess = 0.3
tau_grid = np.linspace(0.0, 1.0, 400, endpoint=False)
chi2_grid = np.array([linear_amplitudes(pbest, e_guess, tau)[3].chi2
                      for tau in tau_grid])
tau_ls = float(tau_grid[np.argmin(chi2_grid)])
K_ls, omega_ls, gamma_ls, _ = linear_amplitudes(pbest, e_guess, tau_ls)
print(f'LS baseline: P={pbest:.2f} d, e={e_guess:.2f}, tau={tau_ls:.3f}, '
      f'K={K_ls:.2f} km/s, omega={omega_ls:.3f} rad, gamma={gamma_ls:.3f} km/s')

In [ ]:
# Overlay the closed-form baseline curve on the data.
def rv_curve(P_days, e, tau, K, omega, gamma, times):
    """Predicted RV via the audited rv_model, parametrized by physical K.

    rv_model takes a companion mass + total mass rather than K, so we map
    K -> projected companion mass using the audited semi-amplitude helper
    (K is linear in mass at fixed P, e, M_total).
    """
    P_yr = P_days / DAYS_PER_YEAR
    k_per_msun = semi_amplitude_kms(mass_msun=1.0, period_yr=P_yr, ecc=e,
                                     M_total_msun=M_TOTAL_MSUN)
    mass_msun = K / k_per_msun
    return rv_model(times, period_yr=P_yr, ecc=e, omega_rad=omega, tau=tau,
                    mass_msun=mass_msun, M_msun=M_TOTAL_MSUN,
                    offset_kms=gamma, epoch_ref_mjd=T_REF)

tt = np.linspace(t.min(), t.max(), 1000)
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(t, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2, label='data')
ax.plot(tt, rv_curve(pbest, e_guess, tau_ls, K_ls, omega_ls, gamma_ls, tt),
        color='C0', lw=1.5, label='LS baseline')
ax.set_xlabel('time (days from J2010.0)')
ax.set_ylabel('radial velocity (km/s)')
ax.legend()
plt.show()

## 4. Quick maximum-likelihood point ('quick and dirty fit')

Now optimise the three *non-linear* shape parameters `(P, e, tau)` with
`scipy.optimize.minimize`, solving the linear amplitudes in closed form
inside the objective. The objective is the audited marginal
log-likelihood ranking score from `linear_solve_rv`. We try a few starts
to avoid obvious local minima.

In [ ]:
def neg_marginal_logL(shape):
    P_days, e, tau = shape
    if not (10.0 < P_days < 500.0 and 0.0 <= e < 0.95 and 0.0 <= tau < 1.0):
        return 1e12
    X = rv_design_matrix(t, period_yr=P_days / DAYS_PER_YEAR, ecc=e,
                          tau=tau, epoch_ref_mjd=T_REF)
    try:
        sol = linear_solve_rv(v, verr, X)
    except Exception:
        return 1e12
    return -sol.logL_marginal

best = None
for e0 in (0.1, 0.45, 0.7):
    for tau0 in np.linspace(0.05, 0.95, 5):
        res = minimize(neg_marginal_logL, [pbest, e0, tau0], method='Nelder-Mead')
        if best is None or res.fun < best.fun:
            best = res

P_ml, e_ml, tau_ml = best.x
K_ml, omega_ml, gamma_ml, _ = linear_amplitudes(P_ml, e_ml, tau_ml)
theta_ml = np.array([P_ml, e_ml, tau_ml, K_ml, omega_ml, gamma_ml])
print('quick ML point (P[d], e, tau, K[km/s], omega[rad], gamma[km/s]):')
print('  P     = %.2f' % P_ml)
print('  e     = %.3f' % e_ml)
print('  tau   = %.3f' % tau_ml)
print('  K     = %.2f' % K_ml)
print('  omega = %.3f' % omega_ml)
print('  gamma = %.3f' % gamma_ml)

## 5. An explicit, editable probability (marginalize the amplitudes)

We now sample **only the three non-linear shape parameters**
`theta = (P, e, tau)`, exactly as `fit_astrometric_orbit.ipynb` and
`fit_joint_orbit.ipynb` do. This is the consistent thing to do: at a
**fixed** shape the RV model is *linear* in the amplitudes
`beta = (gamma, C, S) = (gamma, K cos(omega), K sin(omega))`,

$$ v = \gamma\cdot 1 + (K\cos\omega)(\cos\nu + e) + (K\sin\omega)(-\sin\nu), $$

so the amplitudes can be **integrated out analytically** rather than
sampled. We use the audited linear core both for the marginal evidence
and for the conditional recovery of the amplitudes:

- **Likelihood** = the *marginal* evidence at the shape. Build the design
  with `rv_design_matrix`, solve with `linear_solve_rv`, and return
  `sol.logL_marginal` (the flat-prior marginal-likelihood ranking score
  with the amplitudes integrated out in closed form -- a Gaussian
  evidence that retains the `log|M|` Occam term).
- **Prior** is a simple bounded box over `(P, e, tau)` only.
- **Recovering `(K, omega, gamma)`** per posterior draw: at a fixed
  `(P, e, tau)` the conditional posterior of the amplitudes is the
  Gaussian `N(beta_hat, Sigma_beta) = N(sol.beta, sol.cov)`. We **draw**
  `beta_draw ~ N(sol.beta, sol.cov)` for each sampled shape and map it to
  `K = sqrt(C^2 + S^2)`, `omega = atan2(S, C)`, `gamma = beta_draw[0]`.

Drawing (rather than using `sol.beta` directly) is the whole point: it
carries the amplitudes' **proper conditional uncertainty** into the
corner plot. Using the point estimate `sol.beta` would make
`(K, omega, gamma)` artificially sharp (under-dispersed), because it
would ignore the spread of the conditional Gaussian at each shape.

The likelihood below is the audited `linear_solve_rv` marginal score; the
prior is a simple bounded box you can edit at will. This now matches the
astrometric and joint notebooks, which also sample only `(P, e, tau)` and
recover their linear amplitudes per draw.

In [ ]:
# Box prior bounds on the SHAPE theta = (P, e, tau).
P_LO, P_HI = 100.0, 300.0      # days


def log_prior(theta):
    P, e, tau = theta
    if P_LO < P < P_HI and 0.0 <= e < 0.9 and 0.0 <= tau < 1.0:
        return 0.0
    return -np.inf


def log_likelihood(theta, t, v, verr):
    """Marginal log-likelihood at the shape: amplitudes integrated out.

    At fixed (P, e, tau) the RV model is linear in beta = (gamma, C, S);
    linear_solve_rv returns the flat-prior marginal-likelihood ranking
    score sol.logL_marginal with beta analytically marginalized.
    """
    P, e, tau = theta
    X = rv_design_matrix(t, period_yr=P / DAYS_PER_YEAR, ecc=e,
                         tau=tau, epoch_ref_mjd=T_REF)
    sol = linear_solve_rv(v, verr, X)
    return float(sol.logL_marginal)


def log_probability(theta, t, v, verr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    try:
        ll = log_likelihood(theta, t, v, verr)
    except Exception:
        return -np.inf
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


# theta is now the SHAPE only; build the shape ML point from the quick fit.
theta_ml = np.array([P_ml, e_ml, tau_ml])
print('log_probability at the quick ML shape:',
      log_probability(theta_ml, t, v, verr))

## 6. Sample the posterior with emcee

A bare `emcee.EnsembleSampler` seeded in a tiny Gaussian ball around the
quick ML point. We run a modest chain, discard a burn-in, and look at
the traces and a corner plot.

In [ ]:
n_walkers = 32
n_dim = 3          # we sample only the shape (P, e, tau)
n_steps = 1500

p0 = theta_ml + 1e-3 * rng.standard_normal((n_walkers, n_dim))
p0[:, 1] = np.clip(p0[:, 1], 0.0, 0.89)   # keep e in support
p0[:, 2] = np.clip(p0[:, 2], 0.0, 0.999)  # keep tau in [0, 1)

sampler = emcee.EnsembleSampler(
    n_walkers, n_dim, log_probability, args=(t, v, verr),
)
sampler.run_mcmc(p0, n_steps, progress=True)
print('mean acceptance fraction:', np.mean(sampler.acceptance_fraction))

In [ ]:
labels = ['P (d)', 'e', 'tau']
fig, axes = plt.subplots(n_dim, figsize=(8, 5), sharex=True)
chain = sampler.get_chain()
for i in range(n_dim):
    axes[i].plot(chain[:, :, i], color='k', alpha=0.3, lw=0.5)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel('step')
plt.show()

In [ ]:
burnin = 500
thin = 5
flat_shape = sampler.get_chain(discard=burnin, thin=thin, flat=True)

# Recover (K, omega, gamma) per posterior draw by DRAWING from the
# conditional Gaussian of the linear amplitudes at each sampled shape:
#   beta ~ N(sol.beta, sol.cov)  ->  K = |(C, S)|, omega = atan2(S, C),
#                                     gamma = beta[0].
# Drawing (not using sol.beta) carries the amplitudes' proper conditional
# uncertainty into the corner; a point estimate would under-disperse them.
def recover_amplitudes(flat_shape, rng):
    K_draw = np.empty(flat_shape.shape[0])
    omega_draw = np.empty(flat_shape.shape[0])
    gamma_draw = np.empty(flat_shape.shape[0])
    for j, (P, e, tau) in enumerate(flat_shape):
        X = rv_design_matrix(t, period_yr=P / DAYS_PER_YEAR, ecc=e,
                             tau=tau, epoch_ref_mjd=T_REF)
        sol = linear_solve_rv(v, verr, X)
        beta = rng.multivariate_normal(sol.beta, sol.cov)
        K_draw[j] = recover_K(beta)
        omega_draw[j] = recover_omega(beta) % (2.0 * np.pi)
        gamma_draw[j] = beta[0]
    return K_draw, omega_draw, gamma_draw

K_draw, omega_draw, gamma_draw = recover_amplitudes(flat_shape, rng)

# Full posterior table: sampled shape + recovered amplitudes (with their
# proper conditional dispersion).
flat = np.column_stack([flat_shape, K_draw, omega_draw, gamma_draw])
labels = ['P (d)', 'e', 'tau', 'K (km/s)', 'omega (rad)', 'gamma (km/s)']
fig = corner.corner(flat, labels=labels, show_titles=True,
                    title_fmt='.3f', quantiles=[0.16, 0.5, 0.84])
plt.show()

## 7. Look at the data: phase fold + residuals

Finally, fold the data on the posterior-median period and overlay ~100
posterior-sample model curves, then plot the observed-minus-computed
(O-C) residuals against phase. We also report the 16/50/84 percentiles
for the physical parameters and compare to the injected truth.

In [ ]:
pct = np.percentile(flat, [16, 50, 84], axis=0)
truth = bundle.truth
truth_vals = {'P (d)': truth.P_days, 'e': truth.e, 'K (km/s)': truth.K1_kms,
              'omega (rad)': truth.omega_rad, 'gamma (km/s)': truth.gamma_kms}
print('parameter        16%       50%       84%     truth')
for i, lab in enumerate(labels):
    tv = truth_vals.get(lab, float('nan'))
    print(f'{lab:14s} {pct[0, i]:8.3f}  {pct[1, i]:8.3f}  {pct[2, i]:8.3f}  {tv:8.3f}')

theta_med = pct[1]

In [ ]:
P_med, e_med, tau_med, K_med, omega_med, gamma_med = theta_med

def phase_of(times, P_days, tau):
    tp = tau * P_days + T_REF
    return ((times - tp) / P_days) % 1.0

phase = phase_of(t, P_med, tau_med)
ph_grid = np.linspace(0.0, 1.0, 500)
t_grid = ph_grid * P_med + (tau_med * P_med + T_REF)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

# ~100 posterior-sample curves.
idx = rng.choice(flat.shape[0], size=100, replace=False)
for j in idx:
    P_s, e_s, tau_s, K_s, om_s, g_s = flat[j]
    ax1.plot(ph_grid, rv_curve(P_s, e_s, tau_s, K_s, om_s, g_s, t_grid),
             color='C0', alpha=0.05, lw=1)
ax1.errorbar(phase, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2, zorder=5)
ax1.set_ylabel('radial velocity (km/s)')
ax1.set_title('Phase-folded RV with posterior samples')

# O-C residuals vs phase, against the median-curve model.
v_model_at_obs = rv_curve(P_med, e_med, tau_med, K_med, omega_med, gamma_med, t)
resid = v - v_model_at_obs
ax2.axhline(0.0, color='C3', lw=1)
ax2.errorbar(phase, resid, yerr=verr, fmt='o', color='k', ms=4, capsize=2)
ax2.set_xlabel('orbital phase')
ax2.set_ylabel('O - C (km/s)')
plt.show()

print('residual RMS: %.3f km/s' % np.sqrt(np.mean(resid ** 2)))

## To go further: candidate companion mass (appendix)

*(Optional. An SB1 RV fit measures the orbit and the amplitude `K_kms`, not a
companion mass directly. This appendix turns the posterior into a*
**candidate** *companion mass — it is not a compact-object claim.)*

An SB1 RV orbit constrains the **spectroscopic mass function**

$$f_m \;=\; \frac{(m_2 \sin i)^3}{(m_1 + m_2)^2}
       \;=\; \frac{K^3\,P\,(1-e^2)^{3/2}}{2\pi G},$$

with $K$ the primary semi-amplitude, $P$ the period and $e$ the
eccentricity (all measured here). To turn $f_m$ into a companion mass
$m_2$ we must **assume** two things the spectroscopy alone cannot supply:

- a **primary mass** $m_1$ (e.g. from spectroscopy / isochrones), and
- an **inclination** $\sin i$.

Taking $\sin i = 1$ (edge-on) gives the **minimum** companion mass
consistent with the data — a *lower bound*, not a best estimate. The number
below is a **candidate** companion mass under the dark-companion
($\beta = L_2/L_1 = 0$) assumption: it does **not** by itself establish a
compact object. Luminous-companion, SB2, blend, or triple scenarios would
break this reading, and the true mass needs the inclination — which
**astrometry** supplies (see `fit_astrometric_orbit.ipynb` and the joint
fit). For the convention-tracked library version of this step see
`orblet.interpret.companion_mass.companion_mass_from_rv_posterior`.

In [ ]:
# Candidate companion mass from the open posterior (mass-function inversion).
# Self-contained: we invert fm = (m2 sin i)^3 / (m1 + m2)^2 per posterior
# draw, using the SAME (P, e, K) chain sampled above.
import astropy.units as u
import astropy.constants as const
from scipy.optimize import brentq

# Assumptions (edit freely). m1 from the synthetic catalog; sin i = 1.
# m1 is taken as a fixed scalar here (its uncertainty is neglected) for
# this lower-bound illustration; propagate an m1 prior for a full estimate.
M1_ASSUMED_MSUN = float(bundle.catalog_row["m1"])  # assumed primary mass
SIN_I           = 1.0   # edge-on -> MINIMUM companion mass (lower bound)

# Mass function fm = K^3 P (1-e^2)^{3/2} / (2 pi G), per posterior draw.
# flat columns are (P_days, e, tau, K_kms, omega, gamma).
P_s   = flat[:, 0] * u.day
e_s   = flat[:, 1]
K_s   = np.abs(flat[:, 3]) * (u.km / u.s)
fm_s  = (K_s**3 * P_s * (1.0 - e_s**2)**1.5 / (2.0 * np.pi * const.G)).to(u.Msun)

# Invert fm = (m2 sin i)^3 / (m1 + m2)^2 for m2 at fixed (m1, sin i).
def m2_from_fm(fm_msun, m1_msun, sin_i):
    """Solve the SB1 mass function for the companion mass m2 (M_sun)."""
    def resid(m2):
        return (m2 * sin_i)**3 / (m1_msun + m2)**2 - fm_msun
    # m2 is bracketed in (0, large]; the LHS is monotonic in m2.
    return brentq(resid, 1e-6, 1e4)

m2_s = np.array([m2_from_fm(float(fm), M1_ASSUMED_MSUN, SIN_I)
                 for fm in fm_s.value])

q16, q50, q84 = np.percentile(m2_s, [16, 50, 84])
fm16, fm50, fm84 = np.percentile(fm_s.value, [16, 50, 84])
print(f"mass function fm : {fm50:.3f}  ({fm16:.3f} - {fm84:.3f})  M_sun")
print(f"assumed m1       : {M1_ASSUMED_MSUN:.2f} M_sun   (sin i = {SIN_I:g} -> MINIMUM m2)")
print(f"candidate m2     : {q50:.2f}  ({q16:.2f} - {q84:.2f}, 16/84%)  M_sun")
print("CANDIDATE companion mass under the dark-companion beta=0 assumption;")
print("a lower bound (sin i = 1), NOT a compact-object claim.")

## Summary

We recovered the injected SB1 orbit (typical values from a run of this
notebook: `P ~ 185 d`, `e ~ 0.45`, `K ~ 67 km/s` -- read the table the
cells print, not these) starting from raw radial velocities, using only standard
tools plus audited `orblet` forward-model and likelihood functions.
Every step -- periodogram, closed-form baseline, quick ML, the explicit
prior/likelihood/probability, and the emcee run -- is laid out in the
open so you can edit priors, bounds, and the model freely.

To go further: add a jitter parameter to `theta` (the audited
`rv_loglike` already accepts `jitter_kms`), widen the prior box, or
swap in your own period search. The fully non-linear fit is
`fit_rv_orbit_all_parameters.ipynb`; SEED it from this quick-look
(`compose_rv_seed(...)`): the full fit refines, it does not search.